# Objects, names, and the data model

This is one of the most important conceptual modules in the entire course because it replaces a weak beginner mental model with a precise one. In Python, a variable is not a box holding a value. A **name** is a label bound to an **object**. Multiple names can refer to the same object, and whether changes are shared depends on whether the object is mutable.

That single idea explains aliasing bugs, mutable default argument surprises, confusion around `is` versus `==`, and many copy-related mistakes. Once you see names and objects separately, the behaviour becomes easier to predict.

As you work through the notebook, distinguish carefully between **rebinding a name** and **mutating an object**. Similar syntax can produce very different results.

## Visual model

```text
a -----> [1, 2, 3] <----- b

rebind a:   a -----> [9, 9]
mutate b:   a and b still point to the SAME object
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. Names are not boxes

The most important diagram in this course:

```text
    WRONG mental model              RIGHT mental model

    a: [ 1, 2, 3 ]                  a ──┐
    b: [ 1, 2, 3 ]                      ├──> [ 1, 2, 3 ]   (one object)
                                    b ──┘
```


In [ ]:
a = [1, 2, 3]
b = a              # binds the name b to the SAME object. No copy happens.

b.append(4)        # MUTATES the object
print(a)           # [1, 2, 3, 4]   -- a "changed" without being mentioned
print(a is b)      # True

Now the crucial contrast:

In [ ]:
a = [1, 2, 3]
b = a
b = [9, 9, 9]      # REBINDS the name b to a NEW object
print(a)           # [1, 2, 3]  -- unchanged
print(a is b)      # False

**Mutation changes the object. Rebinding changes the name.** Everyone who has
been bitten by this bug confused the two. Two operations, similar syntax,
opposite consequences:

| Operation | Effect | Visible through other names? |
|---|---|---|
| `b = [9]` | Rebind `b` | No |
| `b.append(9)` | Mutate the object | **Yes** |
| `b += [9]` (list) | Mutate in place (`__iadd__`) | **Yes** |
| `b = b + [9]` | Build new list, rebind | No |

That third and fourth row are the same operation in most languages. In Python
they are not, and the difference is `list.__iadd__` existing. For tuples, which
have no `__iadd__`, `t += (9,)` falls back to `t = t + (9,)` and rebinds.

In [ ]:
def demo() -> None:
    x = [1]; y = x; y += [2];      print(x)   # [1, 2]  mutated
    x = [1]; y = x; y = y + [2];   print(x)   # [1]     rebound

---

## Concept 4. `is` versus `==`

In [ ]:
a is b      # identity: are these the SAME object?         (id(a) == id(b))
a == b      # equality: do these objects COMPARE equal?    (calls __eq__)

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
a == b     # True   -- same contents
a is b     # False  -- two distinct objects

**The rule: use `is` only for singletons.**

In [ ]:
if x is None: ...          # correct, always
if x is True: ...          # correct but usually unnecessary; prefer `if x:`
if flag is Sentinel: ...   # correct for your own sentinel objects
if name is "admin": ...    # WRONG. Use ==. It may appear to work. It is a bug.

Why is `is` wrong for values? Because whether two equal values are the same
object is an implementation detail:

```text
>>> a = 256; b = 256; a is b
True
>>> a = 257; b = 257; a is b
False          # in a REPL. In a single compiled block, possibly True.
```


CPython pre-allocates the integers -5 through 256 at startup and reuses them.
That is **small-int caching**, a CPython optimisation, not a language rule.
String literals get similar treatment (**interning**) for identifier-like
strings. Python 3.8+ emits a `SyntaxWarning` for `is` with a literal, precisely
because this bug was so common.

```text
>>> x = "hello"; y = "hello"; x is y
True                    # both interned
>>> x = "hello world!"; y = "hello world!"; x is y
False                   # not interned (contains characters that make it
                        #  ineligible under the current heuristic)
```


Never build logic on any of this. Use `==` for values, `is` for `None` and
sentinels. Full stop.

### The sentinel pattern

`is` has one genuinely important use beyond `None`: distinguishing "not
provided" from "provided as None".

In [ ]:
_MISSING = object()      # a unique object that equals nothing else

def get(config: dict[str, object], key: str, default: object = _MISSING) -> object:
    if key in config:
        return config[key]
    if default is _MISSING:
        raise KeyError(key)      # caller gave no default: missing is an error
    return default               # caller gave a default, possibly None

You will see this in the standard library and in every serious library. It is
the only way to let `None` be a legitimate default value.

---

## Concept 5. Function arguments: call by object reference

Python is neither "pass by value" nor "pass by reference". Both terms mislead,
and material using them is a reliable signal that the author has not thought
about it. Python passes **object references, by value**. The parameter name is
a new name bound to the same object.

In [ ]:
def rebind(lst: list[int]) -> None:
    lst = [9, 9, 9]        # rebinds the LOCAL name. Caller sees nothing.

def mutate(lst: list[int]) -> None:
    lst.append(9)          # mutates the SHARED object. Caller sees it.

data = [1, 2]
rebind(data);  print(data)      # [1, 2]
mutate(data);  print(data)      # [1, 2, 9]

Same parameter, same call syntax, opposite effect — determined entirely by
whether the body rebinds or mutates.

### The mutable default argument

The most famous Python bug, and now you can explain it rather than memorise it.

In [ ]:
def add_item(item: str, basket: list[str] = []) -> list[str]:
    basket.append(item)
    return basket

print(add_item("apple"))    # ['apple']
print(add_item("pear"))     # ['apple', 'pear']    <-- !

**Default arguments are evaluated once, when the `def` statement executes**, not
on each call. That one list object is stored on the function and reused forever.
You can see it:

```text
>>> add_item.__defaults__
(['apple', 'pear'],)
```


The fix, every time:

In [ ]:
def add_item(item: str, basket: list[str] | None = None) -> list[str]:
    if basket is None:
        basket = []
    basket.append(item)
    return basket

Note `is None`, not `== None` or `if not basket` — an empty list passed
deliberately is falsy and would be silently replaced.

The same trap applies to `{}`, `set()`, and to any expression evaluated at def
time: `def log(t=datetime.now())` freezes the timestamp at import.

`ruff` catches this with rule `B006`, which is enabled in this course's config.

---

## Concept 6. Copying

In [ ]:
import copy

original = [[1, 2], [3, 4]]

alias    = original                 # same object
shallow  = original[:]              # new outer list, SAME inner lists
shallow2 = list(original)           # identical to the above
shallow3 = copy.copy(original)      # identical to the above
deep     = copy.deepcopy(original)  # new outer AND new inner objects

original[0].append(99)
print(alias)     # [[1, 2, 99], [3, 4]]
print(shallow)   # [[1, 2, 99], [3, 4]]   <-- shared inner list
print(deep)      # [[1, 2], [3, 4]]       <-- fully independent

```text
alias    ────────────────> [ ● , ● ]
                             │   │
original ──────────────────> │   │
                             v   v
shallow  ──> [ ● , ● ] ────> [1,2] [3,4]
                ^ ^            ^     ^
                └─┴────────────┴─────┘   (shared!)

deep     ──> [ ● , ● ] ────> [1,2]' [3,4]'   (fresh copies)
```


Practical guidance:

- A shallow copy is enough when the contents are immutable. `list(nums)` for a
  list of ints is genuinely safe.
- `deepcopy` is correct but slow, and it recurses through everything reachable —
  including, by accident, a database connection or a whole object graph.
- The best answer is usually **avoid needing a copy**: use immutable data, or
  return new objects instead of mutating in place. This is the theme that
  Modules 11 and 14 build on.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: Everything is an object
- Section 2: Names are not boxes
- Section 3: Mutable and immutable
- Section 4: `is` versus `==`
- Section 5: Function arguments: call by object reference
- Section 6: Copying
- Section 7: Memory: reference counting and the cycle collector
- Section 8: Truthiness
- Section 9: Namespaces are dictionaries

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import copy
import timeit
from typing import Any

BASE_CONFIG: dict[str, Any] = {
    "service": "api",
    "port": 8080,
    "retries": 3,
    "database": {
        "host": "db.internal",
        "port": 5432,
        "pool": {"min": 2, "max": 10},
        "replicas": ["r1.internal", "r2.internal"],
    },
    "features": {"beta": False, "tracing": True},
    "allowed_origins": ["https://app.example.com"],
}


# TODO 1 -----------------------------------------------------------------------

---

## `derive_shallow`

Shallow copy plus top-level overrides.

In [ ]:
def derive_shallow(base: dict[str, Any], **overrides: Any) -> dict[str, Any]:
    """Shallow copy plus top-level overrides.

    Implement it, then answer in a docstring comment:
      - which mutations of the RESULT would corrupt BASE?
      - which would not?
      - is this ever the right choice? When?
    """
    raise NotImplementedError

---

## `derive_deep`

Full deep copy plus top-level overrides.

In [ ]:
def derive_deep(base: dict[str, Any], **overrides: Any) -> dict[str, Any]:
    """Full deep copy plus top-level overrides.

    Correct, and the obvious answer. Then answer:
      - what does deepcopy do if the structure contains an open file handle,
        a database connection, or a socket?
      - what does it do with a reference cycle?  (Try it. It handles it. How?)
    """
    raise NotImplementedError

---

## `derive_merge`

Recursive merge that never mutates either input and copies only what it

In [ ]:
def derive_merge(base: dict[str, Any], overrides: dict[str, Any]) -> dict[str, Any]:
    """Recursive merge that never mutates either input and copies only what it
    must. Nested dicts merge key by key; lists and scalars are replaced.

        derive_merge(BASE, {"database": {"pool": {"max": 50}}})

    must produce a config where database.pool.min is still 2, database.host is
    unchanged, and BASE is untouched.

    This is what real configuration libraries do, and it is the strategy that
    scales.
    """
    raise NotImplementedError

---

## `freeze`

Return a deeply immutable version of a nested structure.

In [ ]:
def freeze(value: Any) -> Any:
    """Return a deeply immutable version of a nested structure.

    dict  -> a frozen mapping (use types.MappingProxyType, and recurse)
    list  -> tuple (recursing into elements)
    set   -> frozenset
    other -> unchanged

    Then answer: MappingProxyType is a read-only VIEW. What does that mean for
    the underlying dict, and why is this weaker protection than it looks?
    """
    raise NotImplementedError

---

## `measure`

Time all three strategies over 10_000 iterations and print a table.

In [ ]:
def measure() -> None:
    """Time all three strategies over 10_000 iterations and print a table.

    Then answer:
      - what is the ratio between the cheapest and the most expensive?
      - at what call rate would that difference actually matter?
      - and therefore: is 'deepcopy is slow' a reason to avoid it here?
    """
    raise NotImplementedError

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    snapshot = copy.deepcopy(BASE_CONFIG)

    merged = derive_merge(BASE_CONFIG, {"database": {"pool": {"max": 50}}})
    assert merged["database"]["pool"]["max"] == 50
    assert merged["database"]["pool"]["min"] == 2, "merge must not drop siblings"
    assert merged["database"]["host"] == "db.internal"
    assert BASE_CONFIG == snapshot, "derive_merge mutated its input"

    merged["database"]["replicas"].append("r3")
    assert BASE_CONFIG == snapshot, "result shares mutable state with the base"

    print("all checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()
    measure()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.